# SMILES 2026 Submission Runner

Use `MODE_FLAG = '--llm-check'` for the final released solution, or switch to `MODE_FLAG = '--fusion'` to run the secondary 3-head fusion model.

In [ ]:
from pathlib import Path
import shutil
import subprocess
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/olgafilimonova2004/hallucination_detection_SMILES_2026'
REPO_PATH = Path('/content/hallucination_detection_SMILES_2026')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Google Drive output folder:', DRIVE_OUTPUT_DIR)

def run(cmd, cwd=None):
    print('$', cmd)
    process = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'command failed: {cmd}')

if REPO_PATH.exists():
    print('Repository already exists:', REPO_PATH)
    run('git pull', cwd=REPO_PATH)
else:
    run(f'git clone {REPO_URL} {REPO_PATH}')

print('Repository:', REPO_PATH)


In [ ]:
run('pip install -q -r requirements.txt', cwd=REPO_PATH)


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
MODE_FLAG = '--llm-check'  # change to '--fusion' to run the secondary stacked model
run(f'python solution.py {MODE_FLAG}', cwd=REPO_PATH)


In [ ]:
import json
import pandas as pd

results = REPO_PATH / 'results.json'
predictions = REPO_PATH / 'predictions.csv'

for src in (results, predictions):
    if src.exists():
        dst = DRIVE_OUTPUT_DIR / src.name
        shutil.copy2(src, dst)
        print(f'Saved {src.name} to {dst}')
    else:
        print(f'{src.name} not found at {src}')

if results.exists():
    with open(results) as f:
        results_data = json.load(f)
    print('results.json keys:', list(results_data.keys()))
    display(results_data)

if predictions.exists():
    display(pd.read_csv(predictions).head())
